# 03 — Feature Pipeline & Physical Plan Analysis

This notebook runs the full feature pipeline across all 46M rows of M5.
It inspects the Catalyst execution plan, checks for partition skew, validates against `CA_1`, writes the full `features` table, and computes per-series scale and weights per fold.

In [ ]:
# Setup repository path
import sys
REPO = "/Workspace/Users/veysel/m5-forecasting-spark"  # adjust if your repo path differs
if REPO not in sys.path:
    sys.path.append(REPO)

from pyspark.sql import functions as F
from src.features import build_features

base = spark.table("workspace.m5.base")
print(f"Base table row count: {base.count():,}")

## 7.1 Read the Plan Before Running It (`explain`)

Inspect the physical plan to identify:
1. `Exchange hashpartitioning(id, ...)` -> First shuffle (for `id`-based lags and rolling windows)
2. `Sort [id, d_num]` -> In-partition sorting for rolling frames
3. `Window` nodes -> Multiple features sharing the same partition/sort
4. `Exchange hashpartitioning(store_id, dept_id, wm_yr_wk, ...)` -> Second shuffle (from `price_rel_dept`)
5. `BroadcastHashJoin` -> Small calendar table broadcasted without shuffling the 46M row sales table

In [ ]:
feat = build_features(base)
feat.explain(mode="formatted")

## 7.2 Partition Skew Analysis (without RDD API)

Serverless runtimes do not expose the RDD API (`df.rdd.getNumPartitions()`).
We use `F.spark_partition_id()` to check partition distribution: total partitions, min, max, and avg rows per partition.

In [ ]:
(feat.groupBy(F.spark_partition_id().alias("pid")).count()
     .select(F.count("*").alias("partitions"), F.min("count"), F.max("count"), F.avg("count"))
     .show())

## 7.3 Run Pipeline: First on CA_1, Then Full Dataset

### Step 1: Smoke test on single store (CA_1 ~ 4.6M rows)

In [ ]:
import time

t0 = time.time()
(build_features(base.filter("store_id = 'CA_1'"))
   .write.mode("overwrite").saveAsTable("workspace.m5.features_ca1"))
print(f"CA_1 completed in: {time.time() - t0:.1f}s")

### Step 2: Sanity Check Values
Verify that `lag_28` at day $d$ matches `sales` at day $d - 28$ for a sample series.

In [ ]:
sample_check = spark.table("workspace.m5.features_ca1").filter("id = 'HOBBIES_1_001_CA_1' and d_num in (972, 1000)").select("id", "d_num", "sales", "lag_28")
display(sample_check)

### Step 3: Run Full Feature Pipeline (46M rows)
After execution, click on the **View Query Profile** link below the cell to verify the execution graph and shuffles.

In [ ]:
t_full = time.time()
build_features(base).write.mode("overwrite").saveAsTable("workspace.m5.features")
print(f"Full features table completed in: {time.time() - t_full:.1f}s")

## 7.4 Series Statistics Per Fold (Scale & Weight)

Computes RMSSE scale and revenue weights across the 3 rolling cross-validation folds strictly using data up to each cutoff.

In [ ]:
from functools import reduce
from src.config import CUTOFFS
from src.features import series_scale_and_weight

stats = reduce(lambda a, b: a.unionByName(b),
               [series_scale_and_weight(base, c).withColumn("fold", F.lit(f))
                for f, c in CUTOFFS.items()])
stats.write.mode("overwrite").saveAsTable("workspace.m5.series_stats")
print("workspace.m5.series_stats created successfully.")

### Validation of Series Statistics
Weights must sum to ~1 within each fold, scale must be non-negative.

In [ ]:
stats_df = spark.table("workspace.m5.series_stats")
stats_df.groupBy("fold").agg(
    F.sum("weight").alias("total_weight"),
    F.min("scale").alias("min_scale"),
    F.count(F.when(F.col("scale").isNull(), 1)).alias("null_scales")
).show()